# C-test ONE · Unquestionability

*Ctest-Unquestionability-v1.9.14*

Costs every equivalent and rival of every item under one reference machine, with
forward passes only, and finds for each item the shortest prefix $L_u$ at which no rival costs
less than $\widehat{K} + \delta$. Unconditional costs throughout.

| reads | writes |
|---|---|
| `ctest_battery.json`, `ctest_programs.json`, `ctest_calibration.json` | `ctest_pricing_<machine>.json`, and $L_u$ and $\widehat{K}$ onto `ctest_battery.json` |

**Needs** a GPU and, for gated models, a Hugging Face token. The machine is `REF_MACHINE`.

## Running it

**Locally.** From the repository root, install the requirements and open the notebook in Jupyter:

```
pip install -r requirements.txt
jupyter lab notebooks/
```

API keys are read from environment variables (see `.env.example`). On the first local run the
notebook copies `data/` into a `work/` folder and runs there, so the released data are never
modified; set `CTEST_WORK_DIR` to use another folder.

**Colab.** Upload the data files to `MyDrive/ctest` and mount Drive; keys go in the Secrets panel.

**Kaggle.** Attach a dataset holding the data files and set `KAGGLE_DATASET` to its name; keys go in
Add-ons, Secrets. With Kaggle credentials the notebook can also push its results back as a new
version of the dataset.


In [ ]:
# Local runs work in a copy of the data, so that data/ stays exactly as released.
# On Colab and Kaggle this cell does nothing.
import os, glob, shutil
if not (os.path.exists("/kaggle") or os.path.exists("/content")):
    _root = os.getcwd()
    if os.path.basename(_root) == "notebooks":
        _root = os.path.dirname(_root)
    try:
        from dotenv import load_dotenv          # optional: keys from a .env file
        load_dotenv(os.path.join(_root, ".env"))
    except ImportError:
        pass
    WORK_DIR = os.path.abspath(os.environ.get("CTEST_WORK_DIR", os.path.join(_root, "work")))
    os.makedirs(WORK_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(_root, "data", "*")):
        _dst = os.path.join(WORK_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst)
    os.chdir(WORK_DIR)
    print(f"local run: working in {WORK_DIR}")


## 1 · Install


In [ ]:
%pip install -q -U transformers accelerate bitsandbytes

import os

# Only on Colab. Kaggle ships packages that make `import google.colab` succeed,
# so the import is not evidence of anything; the filesystem is.
if os.path.exists("/kaggle"):
    print("on Kaggle: no Drive to mount, and none needed")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as _e:
        print("not on Colab, so no Drive to mount:", _e)


## 1b · Preamble


In [ ]:
import os, sys, re, json, time, glob, math, shutil, random, hashlib, platform
import collections, statistics as st, datetime, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

BUILD = "Ctest-Unquestionability-v1.9.14"
print("=" * 72)
print(f"{BUILD}")
print("=" * 72)

# The filesystem decides, not an import. Kaggle ships packages that make
# `import google.colab` succeed, which sent ENV to "colab" on Kaggle and killed
# the run on the unmounted-Drive guard.
if os.path.exists("/kaggle"):
    ENV = "kaggle"
elif os.path.isdir("/content"):
    ENV = "colab"
else:
    ENV = "local"
# Two separate places, because they are separate things and always were.
DRIVE_FOLDER   = "ctest"        # Colab: /content/drive/MyDrive/ctest
KAGGLE_DATASET = "ctest-one-data"    # Kaggle: the dataset to read from and push to.
                                # A Kaggle dataset is a FOLDER of files, not a
                                # single file.

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
# OUTDIR is a second place to copy finished files to. On Colab that is Drive.
# On Kaggle there is no such place: an attached dataset is READ-ONLY at
# /kaggle/input, and the working directory is the only writable one and is
# already where files are written. Copying a file onto itself raises
# SameFileError, so OUTDIR stays "." and persistence is the dataset push in the
# cell below.
if ENV == "colab" and os.path.isdir("/content/drive/MyDrive"):
    OUTDIR = DRIVE
else:
    OUTDIR = "."
print(f"environment      {ENV}   python {platform.python_version()}")

# shutil.copy writes a FILE when its destination is not an existing directory, so
# without this every save lands in a file named after the folder and the results
# leave with the session.
if OUTDIR != ".":
    if os.path.exists(OUTDIR) and not os.path.isdir(OUTDIR):
        raise SystemExit(f"{OUTDIR} exists but is a file. Rename or delete it.")
    os.makedirs(OUTDIR, exist_ok=True)
elif ENV == "colab":
    raise SystemExit("Drive is not mounted, so anything written would be lost. Run\n"
                     "    from google.colab import drive; drive.mount('/content/drive')")
if ENV == "kaggle":
    print(f"outputs go to    the working directory, then pushed to the "
          f"{KAGGLE_DATASET} dataset")
    print( "                 (a dataset cannot be written to in place; see the "
           "persistence cell)")
if ENV != "kaggle":
    print(f"outputs go to    {OUTDIR}")

HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        _p = torch.cuda.get_device_properties(0)
        print(f"GPU              {_p.name}, {_p.total_memory/2**30:.0f} GiB")
    else:
        print("GPU              none found. This notebook prices programs under a")
        print("                 reference machine, so it will be very slow on CPU.")
except Exception:
    print("GPU              torch not installed")

def _secret(n):
    """Look for a secret, and SAY what each provider reported.

    An earlier version swallowed the Kaggle error and fell through to the Colab
    branch, which raises on Kaggle, so the result was a bare None and nothing
    said why. Kaggle secrets are PER NOTEBOOK: one attached to another session
    does not carry over."""
    if os.environ.get(n):
        return os.environ[n]
    notes = []
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(n)
        if v:
            return v
        notes.append("kaggle: returned nothing")
    except ImportError:
        pass                                  # not on Kaggle, which is fine
    except Exception as e:
        notes.append(f"kaggle: {type(e).__name__}: {str(e)[:90]}")
    try:
        from google.colab import userdata
        v = userdata.get(n)
        if v:
            return v
        notes.append("colab: returned nothing")
    except ImportError:
        pass
    except Exception as e:
        notes.append(f"colab: {type(e).__name__}: {str(e)[:90]}")
    for m in notes:
        print(f"                 {m}")
    return None

_key = _secret("GOOGLE_API_KEY")
if _key: os.environ["GOOGLE_API_KEY"] = _key
print(f"GOOGLE_API_KEY   {'found' if _key else 'not needed by this notebook'}")

# The dataset, from a secret when there is one, so two accounts can work in
# parallel on separate copies with this notebook byte-identical in both.
_ds = _secret("KAGGLE_DATASET")
if _ds:
    KAGGLE_DATASET = _ds.strip()
print(f"KAGGLE_DATASET   {KAGGLE_DATASET}"
      + ("   from the secret" if _ds else "   the built-in default"))

# The Hub token has to be put INTO the environment before transformers is asked
# for a model. A Kaggle secret is not an environment variable: it lives behind
# UserSecretsClient and is invisible to huggingface_hub until something fetches
# it, which is why a notebook with HF_TOKEN in its secrets still reports
# "unauthenticated requests to the HF Hub". Unauthenticated still works for a
# public model; it is rate limited and slower, and it fails outright on a gated
# one.
#
# Both names are set. huggingface_hub reads HF_TOKEN, and some versions of
# transformers and datasets still look for HUGGING_FACE_HUB_TOKEN.
_hf = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN") \
      or _secret("HUGGINGFACE_TOKEN")
if _hf:
    os.environ["HF_TOKEN"] = _hf
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = _hf
    print(f"HF_TOKEN         found, {len(_hf)} chars, set for the Hub")
else:
    print( "HF_TOKEN         not found. Downloads are unauthenticated: rate")
    print( "                 limited, and gated models will not resolve. On")
    print( "                 Kaggle add it under Add-ons -> Secrets with the")
    print( "                 toggle ON for this notebook; on Colab use the key")
    print( "                 panel. The secret must be named HF_TOKEN.")

_PRINT_LOCK = threading.Lock()

def log(*a, **kw):
    with _PRINT_LOCK:
        print(*a, flush=True, **kw)

def stage(t):
    with _PRINT_LOCK:
        print(f"\n{'='*72}\n== {t}\n{'='*72}", flush=True)

def find(f):
    for p in [f, f"/content/{f}", f"{DRIVE}/{f}"] + glob.glob(f"/kaggle/input/**/{f}", recursive=True):
        if os.path.exists(p): return p
    return None

def load(name, default):
    p = find(name)
    if not p:
        log(f"  {name}: absent, starting empty"); return default
    log(f"  {name}: loaded from {p}")
    return json.load(open(p))


## 2 · Config

`REF_MACHINE` decides which file is written.
`DELTA` is the rival budget in
bits and `L0` the shortest prefix a rival was asked to match; both follow the
values already used in the pipeline.


In [ ]:
REF_MACHINE = "Qwen/Qwen3-14B"
REF_TAG     = ""                   # file suffix; derived from REF_MACHINE if empty
LOAD_4BIT   = True                 # an 8B model does not fit a 16 GiB card in bf16

# delta is NOT set here. It is bounded by Definition 4.1 (delta must be below
# the cost of the cheapest exception clause) and that bound is machine-relative,
# so it is measured against the loaded machine in the calibration cell further
# down. This provisional value exists only so the config can print something;
# the calibration cell overwrites it before the walk runs.
DELTA       = 15.0                 # provisional, see the calibration cell
L0          = 5                    # shortest prefix a rival was asked to match
L_MAX_CAP   = 0
K_ADMIN     = 12                   # terms left for the question, so the highest
                                   # prefix worth considering is len(terms) - 12.
                                   # Matches the generator, which sized every
                                   # sequence by the same rule.

CERTIFY_AT_FALLBACK = [5, 13]      # for records written before asked_at existed

BETA_LADDER_N = 6                  # points on the ladder, spread over the range

PRICE_PREFIX = "v17"
PRICE_ONLY = None
                                   # item the run has not judged yet. None is
                                   # what is wanted here: REPRICE is off, so
                                   # the 88 already judged under this machine
                                   # are skipped and the thirteen new ones are
                                   # what is left. Naming them would only give
                                   # the list a chance to be wrong.
MAX_ITEMS  = 0         # no cap
REPRICE    = False     # leave the 41 it has already judged alone
SKIP_EXCLUDED = True               # do not spend GPU on items already marked
TIME_BUDGET_MIN = 690     # stop cleanly before a 12-hour session ends; 0 for none

MARK_QUESTIONABLE = False
MARK_VALUE  = "questionable"       # or "stale", if you prefer the older label

INCLUDE_AUTHORED_RULE = True       # the seed passed determinism at generation,
                                   # so it is a known-good program and belongs
                                   # in the minimisation
WRITE_TO_BATTERY = True            # also put n_M and K on each item, suffixed
                                   # by machine tag, so downstream notebooks do
                                   # not have to open the pricing file
BATCH_ROWS   = 8                   # programs per forward pass
BATCH_TOKENS = 3000                # and a cap on their total length, since the
                                   # scored slice is vocabulary-wide
STRUCT_PRICE_ALL = True            # price all three structural rivals at each
                                   # prefix and take the cheapest
BETA_MIN_R2 = 0.80                 # below this the structural rival's cost does
                                   # not grow cleanly enough with L for an
                                   # extrapolation to mean anything
EXTRAPOLATE_L = True               # when no prefix certifies, estimate the L
                                   # that would, from how the structural rival
                                   # grows

PRICE_TPL = ("Here is the beginning of a sequence:\n{terms}\n"
             "A rule that generates it:")
PRINT_TPL = "Print: "

BATTERY  = "ctest_battery.json"
PROGRAMS = "ctest_programs.json"
# ===========================================================================

def machine_tag(name):
    return re.sub(r"[^a-z0-9.-]+", "-", name.split("/")[-1].lower()).strip("-")

TAG = REF_TAG or machine_tag(REF_MACHINE)
PRICING = f"ctest_pricing_{TAG}.json"

print(f"\nreference machine {REF_MACHINE}")
print(f"file tag          {TAG}  ->  {PRICING}")
print(f"window            delta set later, by the calibration cell")
print(f"certify at        per item, from the prefixes the composer was asked at"
      f" (fallback {CERTIFY_AT_FALLBACK})")
print(f"beta ladder       {BETA_LADDER_N} points spread from {L0} to the item's reach")
print(f"reach             len(terms) - {K_ADMIN}"
      + (f", capped at {L_MAX_CAP}" if L_MAX_CAP else ", uncapped"))
print(f"budget            {TIME_BUDGET_MIN} min"
      + (f", at most {MAX_ITEMS} items" if MAX_ITEMS else ", no item limit"))
print(f"frame             c(p), unconditional. See the notes above.")
print(f"K candidates      verified equivalents"
      + (" plus the authored rule" if INCLUDE_AUTHORED_RULE else ""))
print(f"marking           " + (f"on, as {MARK_VALUE!r}" if MARK_QUESTIONABLE
      else "OFF - look at a run before letting it mark anything"))
if PRICE_ONLY:
    print(f"queue             {len(PRICE_ONLY)} named items")
if MARK_VALUE not in ("questionable", "stale", "flawed"):
    raise SystemExit(f"MARK_VALUE {MARK_VALUE!r} is not one of the recognised values")


## 1c · Persistent state on Kaggle

Reading needs nothing: the loaders glob `/kaggle/input/**` and find the files in
whatever dataset is attached. **Writing** is the problem.
A dataset cannot be
written to directly, and the working directory is wiped between versions, so a
run's output is lost unless it is pushed back as a new version of the dataset.

On Colab this cell does nothing, because Drive is already persistent.

### Setting it up, once

1.
Add Input, Datasets, add **ctest-one-data**.
A Kaggle dataset is a folder of
   files, and adding it makes them readable under `/kaggle/input/ctest-one-data/`.
2.
Add-ons, Secrets: add `KAGGLE_USERNAME` and `KAGGLE_KEY` from your
   `kaggle.json`, and turn the toggle on for this notebook.

With no credentials the notebook still runs and the results still appear in the
version output.
Only state across versions is lost, and the cell says so rather
than failing.

### Why the push works by pattern

`kaggle datasets version` replaces the **whole** dataset with the contents of
the pushed folder, so a file that is in the dataset but not staged is deleted.

So everything matching `ctest_*.json` and `ctest_*.csv` is staged. The log says which were carried.

### When it pushes

A push uploads the whole state, so it does not happen per item.
It happens at
the end of a run, and every `STATE_PUSH_EVERY` items if that is set above zero,
which is worth doing on a long GPU run that might be cut off.


In [ ]:
import subprocess

STATE_PATTERNS = ("ctest_*.json", "ctest_*.csv")

STATE_DATASET = ""        # "owner/slug"; empty derives it from KAGGLE_DATASET
STATE_PUSH_EVERY = 5

def _kaggle_ready():
    if ENV != "kaggle":
        return False
    missing = []
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        v = _secret(k)
        if v:
            os.environ[k] = v
        else:
            missing.append(k)
    if missing:
        log(f"  state persistence OFF, missing secret(s): {', '.join(missing)}")
        log("     Add-ons -> Secrets -> add them from your kaggle.json, and turn")
        log("     the toggle on for this notebook. Results still appear in the")
        log("     version output; only state across versions is lost.")
        return False
    return True

KAGGLE_OK = _kaggle_ready()

def _slug():
    if STATE_DATASET:
        return STATE_DATASET
    return f"{os.environ.get('KAGGLE_USERNAME','unknown')}/{KAGGLE_DATASET.lower()}"

def state_files():
    """Every file the push should carry, newest copy of each name.

    Local first, since this run may have just written it. Then whatever is in
    the attached dataset and not local, so another notebook's output survives a
    push from this one."""
    chosen, source = {}, {}
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(pat)):
            chosen[os.path.basename(f)] = f
            source[os.path.basename(f)] = "this run"
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True)):
            b = os.path.basename(f)
            if b not in chosen:
                chosen[b] = f
                source[b] = "carried forward"
    return chosen, source

FRESH_DIR = "/kaggle/working/_fresh"

def _dataset_names():
    """Filenames actually IN the dataset, from the refreshed download.

    NOT from /kaggle/input. The mount is fixed when the session starts, so it
    can be older than the dataset, and comparing the mount against a push
    assembled from the mount compares a thing with itself and can never fire.
    That was the whole failure: the dataset held ctest_metrics.json, the mount
    did not, and a push assembled from the mount deleted it.

    If the refresh did not run or failed, this returns nothing and the guard
    stays quiet, because there is no trustworthy list to compare against.
    """
    out = set()
    if not os.path.isdir(FRESH_DIR):
        return out
    for pat in STATE_PATTERNS:
        for f in glob.glob(os.path.join(FRESH_DIR, pat)):
            out.add(os.path.basename(f))
    return out

STATE_REFRESH = True

def state_refresh():
    """Download the current dataset version into the working directory.

    Cheap, a couple of megabytes, and it removes a whole class of confusion:
    after this the notebook is reading what is IN the dataset rather than what
    happened to be mounted.

    Never raises. A failure leaves /kaggle/input in place, which is the old
    behaviour, and says so.
    """
    if not (STATE_REFRESH and KAGGLE_OK):
        return
    slug = _slug()
    d = FRESH_DIR
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    try:
        r = subprocess.run(["kaggle", "datasets", "download", "-d", slug,
                            "-p", d, "--unzip"],
                           capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            log(f"  could not refresh from {slug}: "
                f"{(r.stderr or r.stdout).strip()[:120]}")
            log(f"  falling back to whatever the session mounted, which may be "
                f"older than the dataset.")
            return
    except Exception as e:
        log(f"  could not refresh from {slug}: {str(e)[:100]}")
        return

    got, stale = [], []
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(os.path.join(d, pat))):
            b = os.path.basename(f)
            mounted = None
            for p in glob.glob(f"/kaggle/input/**/{b}", recursive=True):
                mounted = p; break
            if os.path.exists(b):
                continue
            shutil.copy(f, b)
            got.append(b)
            if mounted is None:
                stale.append(f"{b} (NOT mounted at all)")
            elif os.path.getsize(mounted) != os.path.getsize(f):
                stale.append(f"{b} (mounted copy is a different size)")
    log(f"  refreshed {len(got)} file(s) from {slug}: {', '.join(sorted(got))}")
    if stale:
        log(f"  the session's mount was OUT OF DATE for: {', '.join(stale)}")
        log(f"  That is why this matters: without the refresh those would have "
            f"been read stale or not at all.")
def state_save(msg="update"):
    """Push a new version of the state dataset. Never raises."""
    if not KAGGLE_OK:
        return
    chosen, source = state_files()
    if not chosen:
        log("  no state files to push yet"); return
    d = "/kaggle/working/_state"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    for b, f in chosen.items():
        shutil.copy(f, os.path.join(d, b))
    _in_dataset = _dataset_names()
    if not _in_dataset:
        log(f"  note: no refreshed copy of the dataset, so the push cannot be "
            f"checked for deletions. Set STATE_REFRESH = True.")
    _would_lose = _in_dataset - set(chosen)
    if _would_lose:
        log(f"  REFUSING TO PUSH. These are in the dataset and not in this "
            f"push, so the push would DELETE them:")
        for b in sorted(_would_lose):
            log(f"      {b}")
        log(f"  Most likely the session mounted an old version of the dataset. "
            f"STATE_REFRESH pulls the current one at startup; if that failed,")
        log(f"  the log above says why. Nothing has been pushed and nothing is "
            f"lost.")
        return

    carried = [b for b, s in source.items() if s == "carried forward"]
    log(f"  staging {len(chosen)} files"
        + (f", {len(carried)} carried forward from the dataset: "
           f"{', '.join(sorted(carried)[:4])}"
           f"{' ...' if len(carried) > 4 else ''}" if carried else ""))
    slug = _slug()
    json.dump({"title": slug.split("/")[-1], "id": slug,
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{d}/dataset-metadata.json", "w"))
    try:
        r = subprocess.run(["kaggle", "datasets", "version", "-p", d, "-m", msg,
                            "-r", "zip", "--dir-mode", "zip"],
                           capture_output=True, text=True, timeout=900)
        out = (r.stdout + r.stderr).strip()
        last = out.split("\n")[-1][:160] if out else ""
        if r.returncode == 0 and "error" not in out.lower():
            log(f"  state push: {last}")
        else:
            log(f"  the version push did not go through. Its output was:")
            for _l in out.split("\n")[-6:]:
                if _l.strip(): log(f"    {_l.strip()[:150]}")
            # Only create when the dataset really is absent. "Already in use"
            # means it exists and the version push failed for some other
            # reason, so creating it cannot help.
            missing = ("404" in out or "not found" in out.lower()) and \
                      "already in use" not in out.lower()
            if missing:
                r2 = subprocess.run(["kaggle", "datasets", "create", "-p", d,
                                     "-r", "zip", "--dir-mode", "zip"],
                                    capture_output=True, text=True, timeout=900)
                o2 = (r2.stdout + r2.stderr).strip()
                if "already in use" in o2.lower():
                    log(f"  the dataset exists after all, so the version push "
                        f"failed for another reason. NOTHING WAS PUSHED: "
                        f"download the files from this version's output.")
                else:
                    log(f"  created dataset '{slug}'. Attach it as an input so "
                        f"the next version can read it back.")
                    log(f"  state push: {o2.split(chr(10))[-1][:160]}")
            else:
                log(f"  NOTHING WAS PUSHED. The files are in this version's "
                    f"output and can be downloaded from there.")
    except Exception as e:
        log(f"  state push failed ({str(e)[:80]}). The results are still in this"
            f" version's output.")

stage("PERSISTENT STATE")
if ENV != "kaggle":
    log(f"  not on Kaggle, so nothing to do. Outputs go to {OUTDIR}.")
elif KAGGLE_OK:
    subprocess.run(["pip", "install", "-q", "kaggle"], capture_output=True)
    log(f"  credentials found for '{os.environ['KAGGLE_USERNAME']}'")
    log(f"  state dataset: {_slug()}")
    log(f"  pushes" + (f" every {STATE_PUSH_EVERY} items and" if STATE_PUSH_EVERY
        else "") + " at the end of the run")
_attached = sorted({os.path.basename(os.path.dirname(f))
                    for f in glob.glob("/kaggle/input/*/*")})
if ENV == "kaggle":
    if _attached:
        log(f"  datasets attached: {', '.join(_attached)}")
    else:
        log(f"  NO dataset attached. Add Input -> Datasets -> {KAGGLE_DATASET},")
        log(f"  or the loaders will not find the battery.")
_ch, _src = state_files()
if _ch:
    log(f"  state would carry {len(_ch)} files: {', '.join(sorted(_ch)[:6])}"
        + (" ..." if len(_ch) > 6 else ""))

state_refresh()


## 1d · What the run cost

One row per run, appended to `ctest_runs.csv`.
Wall clock, CPU seconds, GPU
seconds and device, and tokens by model split into prompt, answer and thinking.

Nothing here is a measurement of energy.
It is the record from which energy and
CO2 can be worked out later, from published per-token figures and device power,
so those numbers can be revised without re-running anything.
Keeping the inputs
and deriving the rest is the only way that stays honest.

The numbers are approximate on purpose.
CPU time is process time, so it counts
this notebook and not the API's work.
GPU seconds are wall-clock while a model
is loaded, not utilisation.
Thinking tokens are separated because they are most
of the output on a reasoning model and are charged the same, and they are why a
call truncates with nothing usable returned.


In [ ]:
import csv, resource

RUNS_CSV = "ctest_runs.csv"

# Seed the local copy from wherever the file already lives, before anything is
# appended. Every other file is read back through find(), which globs
# /kaggle/input; this one was only ever appended, so on Kaggle it started empty
# each session and each run wrote a fresh one-row file. Worse, the dataset push
# prefers the local copy, so that one row REPLACED the accumulated history on
# every push. Rows were being lost, not merely not added.
_prev = find(RUNS_CSV)
if _prev and os.path.abspath(_prev) != os.path.abspath(RUNS_CSV):
    shutil.copy(_prev, RUNS_CSV)
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) carried in from {_prev}")
elif _prev:
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) already here")
else:
    print(f"run history       none yet, {RUNS_CSV} will be created")

_T_START = time.time()
_RUN_KEY = datetime.datetime.now().isoformat(timespec="seconds")
_UNITS_SO_FAR = [0]        # work finished, updated as the run goes
_RUN_NOTE = [""]           # what the row should say
_CPU_START = resource.getrusage(resource.RUSAGE_SELF).ru_utime + \
             resource.getrusage(resource.RUSAGE_SELF).ru_stime
_GPU_SECONDS = [0.0]      # wall clock with a model loaded; add to it when one is

def _gpu_name():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return ""

def account(units=0, note=""):
    """Record this run's cost. Never raises: accounting must not stop work.

    UPSERT, not append. The row is keyed on when the run STARTED, so calling
    this repeatedly updates one row instead of adding several, and the numbers
    only grow. It used to be called once at the end, so a run stopped by hand
    or by the session ending wrote nothing at all and its compute vanished from
    the record.
    """
    try:
        cpu = (resource.getrusage(resource.RUSAGE_SELF).ru_utime +
               resource.getrusage(resource.RUSAGE_SELF).ru_stime) - _CPU_START
        tk = dict(TOKENS) if "TOKENS" in globals() else {}
        row = dict(
            run_started=_RUN_KEY,
            run_ended=datetime.datetime.now().isoformat(timespec="seconds"),
            build=BUILD, env=ENV, note=note,
            wall_s=round(time.time() - _T_START, 1),
            cpu_s=round(cpu, 1),
            gpu_s=round(_GPU_SECONDS[0], 1),
            gpu=_gpu_name(),
            units=units,
            model=(COMPOSER if "COMPOSER" in globals()
                   else REF_MACHINE if "REF_MACHINE" in globals() else ""),
            api_calls=tk.get("calls", 0),
            prompt_tokens=tk.get("in", 0),
            answer_tokens=tk.get("answer", 0),
            thinking_tokens=tk.get("think", 0),
            output_tokens=tk.get("out", 0),
            forward_passes=(REF.calls if "REF" in globals()
                            and hasattr(REF, "calls") else 0),
        )
        old_rows, old_cols = [], []
        if os.path.exists(RUNS_CSV):
            with open(RUNS_CSV, newline="") as f:
                rd = csv.DictReader(f)
                old_cols = list(rd.fieldnames or [])
                old_rows = list(rd)
        old_rows = [r for r in old_rows
                    if r.get("run_started") != _RUN_KEY]
        cols = old_cols + [c for c in row if c not in old_cols]
        with open(RUNS_CSV, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, restval="")
            w.writeheader()
            for r in old_rows + [row]:
                w.writerow({c: r.get(c, "") for c in cols})
        if OUTDIR != ".":
            shutil.copy(RUNS_CSV, OUTDIR)
    except Exception as e:
        log(f"  accounting failed ({str(e)[:70]}), which changes nothing else")


## 3 · Files and the exclusion vocabulary

The battery and the programs are read.
The battery is written back only to add a
mark, so terms, rules and provenance are untouched.


In [ ]:
# Nothing is ever deleted. An item that should not be used is MARKED, so the
# decision stays visible, reversible and auditable. This vocabulary comes from
# the v5 pipeline:
#   excluded = null           in use
#            = "duplicate"    another item measures the same thing
#            = "quota"        bucket full, held back for balance, nothing wrong
#            = "flawed"       defective: ambiguous rule, wrong ground truth
#            = "stale"        measured by an older pipeline, re-measure before use
#            = "questionable" a cheap diverging rival survives at every prefix
# An exclusion already on an item is never overwritten, so a decision made by
# hand always beats an automatic one.
EXCLUSION_VALUES = ("duplicate", "quota", "flawed", "stale", "questionable")

def usable(b):
    """In use, unless something says otherwise."""
    return not b.get("excluded")

def seed_of(b):
    """The rule the composer first wrote. Called `seed` since generator v2.17,
    `rule` before it, and both may be in one battery."""
    return b.get("seed") or b.get("rule") or ""

def cell_of(b):
    return (b.get("want_retrieval"), b.get("want_transformation"))

def mark(b, value, note):
    """Mark an item, unless it already carries a decision."""
    if b.get("excluded"):
        return False
    b["excluded"] = value
    b["excluded_note"] = note
    b["excluded_by"] = BUILD
    b["excluded_on"] = datetime.date.today().isoformat()
    return True

stage("LOADING")
BAT  = load(BATTERY, {"items": []})
PROG = load(PROGRAMS, {"items": {}})
PRICED = load(PRICING, {"_README":
    "Pricing of the stored programs under one reference machine. n_M is the "
    "shortest prefix with no affordable diverging rival. tested_at says where "
    "rivals actually existed, because certifying a prefix nothing was proposed "
    "against is a statement about the search and not about the item.",
    "machine": REF_MACHINE, "machine_tag": TAG, "delta": DELTA, "items": {}})

ITEMS = BAT.setdefault("items", [])
PR = PROG.setdefault("items", {})
DONE = PRICED.setdefault("items", {})

log(f"\n  {len(ITEMS)} items in the battery")
log(f"  {sum(1 for b in ITEMS if usable(b))} usable, "
    f"{dict(collections.Counter(b['excluded'] for b in ITEMS if b.get('excluded')))}")
log(f"  {len(PR)} have a programs record")
log(f"  {len(DONE)} already priced under {TAG}")

_riv = sum(1 for b in ITEMS if PR.get(b["id"], {}).get("rivals"))
log(f"  {_riv} have at least one rival. The rest cannot be tested, only "
    f"certified by default.")

_LOCK = threading.Lock()
def save():
    with _LOCK:
        PRICED["machine"] = REF_MACHINE
        PRICED["machine_tag"] = TAG
        PRICED["delta"] = DELTA
        PRICED["priced_by"] = BUILD
        for name, obj in ((PRICING, PRICED), (BATTERY, BAT)):
            json.dump(obj, open(name, "w"), indent=1)
            if OUTDIR != ".": shutil.copy(name, OUTDIR)

def require(condition, what, *hints):
    """Stop with something actionable, rather than carrying on empty."""
    if condition:
        return
    log("")
    log("=" * 72)
    log(f"STOPPING: {what}")
    log("=" * 72)
    for h in hints:
        log(f"  {h}")
    if ENV == "kaggle":
        seen = sorted(glob.glob("/kaggle/input/*/*"))
        if seen:
            log("")
            log(f"  What IS attached, {len(seen)} file(s):")
            for f in seen[:12]:
                log(f"    {f}")
            if len(seen) > 12:
                log(f"    ... and {len(seen)-12} more")
            log("")
            log("  If the names differ from what this notebook expects, the")
            log("  dataset is the wrong one or the files were renamed.")
        else:
            log("")
            log(f"  Nothing is attached at all. Add Input -> Datasets -> "
                f"{KAGGLE_DATASET}.")
    raise SystemExit(what)

require(len(ITEMS) > 0, "the battery is empty, so there is nothing to price",
        "The battery is read, never written, by this notebook. An empty one",
        "means the file was not found, not that the battery has no items.",
        f"Expected: {BATTERY}")
require(len(PR) > 0, "no item has any programs, so nothing can be priced",
        "Unquestionability needs the equivalents and rivals the generator",
        "wrote. Run the generator with DO_PROGRAMS = True first.",
        f"Expected: {PROGRAMS}")


Optional, and normally skipped.

This works because a pricing pass only ever **adds** to the battery.
It never
touches terms, rules, glosses, programs or provenance.
So clearing the four
`excluded*` fields and the fields suffixed with the machine tag restores the
item exactly, and there is no need to keep a backup copy of the battery or to
regenerate anything.

Marks made by a different build are left alone.
A `flawed` mark from the
generator, or one made by hand, is a different decision and survives.


In [ ]:
UNDO_PASS = None

STALE_FIELDS = ("h_", "prop41_slack_", "untested_", "K_", "slack_",
                "L_needed_", "struct_only_", "n_M_")

if UNDO_PASS:
    n_marks = n_fields = 0
    for b in ITEMS:
        if b.get("excluded_by") == UNDO_PASS:
            for k in ("excluded", "excluded_note", "excluded_by", "excluded_on"):
                b.pop(k, None)
            n_marks += 1
        for k in [k for k in list(b) if any(k.startswith(f) and k.endswith(TAG)
                                            for f in STALE_FIELDS)]:
            b.pop(k, None); n_fields += 1
    log(f"  undid {n_marks} marks from {UNDO_PASS}")
    log(f"  removed {n_fields} stale per-machine fields for {TAG}")
    log(f"  {sum(1 for b in ITEMS if usable(b))} of {len(ITEMS)} items usable now")
    left = collections.Counter(b["excluded"] for b in ITEMS if b.get("excluded"))
    if left:
        log(f"  marks left, made by other builds or by hand: {dict(left)}")
    DONE.clear()
    log(f"  cleared the pricing records for {TAG}; this run reprices from scratch")
    save()
else:
    log("  UNDO_PASS is None, so nothing is undone")


## 4 · The reference machine

It prices strings and does nothing else.
No sampling, no generation.


In [ ]:
class Machine:
    """The reference machine. Prices strings, and nothing else."""

    def __init__(self, name, four_bit=True):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.torch, self.name, self.calls = torch, name, 0
        log(f"  loading {name}" + ("  (4-bit)" if four_bit else "  (bfloat16)"))
        t0 = time.time()
        self.tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        kw = dict(trust_remote_code=True, device_map="auto")
        if four_bit:
            try:
                from transformers import BitsAndBytesConfig
                kw["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
            except Exception as e:
                log(f"  4-bit unavailable ({str(e)[:60]}), using bfloat16")
                kw["torch_dtype"] = torch.bfloat16
        else:
            kw["torch_dtype"] = torch.bfloat16
        self.model = AutoModelForCausalLM.from_pretrained(name, **kw)
        self.model.eval()
        self.dev = next(self.model.parameters()).device
        self._empty, self._memo = None, {}
        self._loaded_at = time.time()   # for the run accounting
        self.build = BUILD          # so a stale instance is not reused
        log(f"  loaded in {time.time()-t0:.0f}s on {self.dev}")

        places = collections.Counter(str(p.device) for p in self.model.parameters())
        log(f"  parameters by device: {dict(places)}")
        if any(d.startswith("cpu") or d == "meta" for d in places):
            log("  WARNING: part of the model is not on the GPU. Every forward")
            log("  pass will be slow. Use a smaller model, or LOAD_4BIT with a")
            log("  card that fits it.")

        # one timed pass, so the rate is known before 36 items are queued
        _t = time.time()
        _b = self.bits("the quick brown fox jumps over the lazy dog\n")
        log(f"  a short pass takes {time.time()-_t:.2f}s "
            f"({_b:.0f} bits for that probe)")

    def _ids(self, s):
        return self.tok(s, add_special_tokens=True).input_ids

    def _score(self, ids_batch, mask, starts):
        """Summed log2 P for each row, from `starts[r]` to that row's end.

        Two things matter for speed. Only the positions being scored are turned
        into probabilities, so the vocabulary-wide softmax is not computed for
        every token in the string. And the whole row is gathered in one
        operation with a single device-to-host transfer, instead of one
        `.item()` per token, which was a separate GPU sync each time."""
        torch = self.torch
        with torch.no_grad():
            logits = self.model(ids_batch, attention_mask=mask).logits
        out = []
        for r in range(ids_batch.shape[0]):
            n0, n1 = starts[r], int(mask[r].sum().item())
            if n1 <= n0:
                out.append(0.0); continue
            lg = logits[r, n0-1:n1-1, :].float()        # only what is scored
            tgt = ids_batch[r, n0:n1]
            lp = torch.log_softmax(lg, dim=-1)
            tot = lp.gather(1, tgt.unsqueeze(1)).sum().item()   # one transfer
            out.append(-tot / math.log(2))
        self.calls += ids_batch.shape[0]
        return out

    def bits_many(self, texts, condition=""):
        """-log2 P(text | condition) for several texts, batched.

        The texts are independent, so one forward pass covers a batch. Right
        padding is safe under causal attention: a token is never influenced by
        anything after it."""
        torch = self.torch
        if not texts:
            return []
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token
        ctx = self._ids(condition) if condition else self._ids("")
        rows, starts = [], []
        for t in texts:
            full = self._ids(condition + t)
            n = len(ctx)
            if full[:n] != ctx:                  # tokenisation did not line up
                n = 0
                while n < min(len(ctx), len(full)) and full[n] == ctx[n]:
                    n += 1
            rows.append(full); starts.append(max(n, 1))
        out, i = [], 0
        while i < len(rows):
            # keep batch * length bounded, since the scored slice is vocab-wide
            batch, tok_budget = [], 0
            while i < len(rows) and len(batch) < BATCH_ROWS:
                L = len(rows[i])
                if batch and tok_budget + L > BATCH_TOKENS:
                    break
                batch.append(i); tok_budget += L; i += 1
            width = max(len(rows[b]) for b in batch)
            pad = self.tok.pad_token_id
            ids = torch.full((len(batch), width), pad, dtype=torch.long)
            mask = torch.zeros((len(batch), width), dtype=torch.long)
            for k, b in enumerate(batch):
                r = rows[b]
                ids[k, :len(r)] = torch.tensor(r)
                mask[k, :len(r)] = 1
            out += self._score(ids.to(self.dev), mask.to(self.dev),
                               [starts[b] for b in batch])
        return out

    def bits(self, text, condition=""):
        """-log2 P(text | condition). The leading space of `text` belongs to
        `text`, so `condition` must not end in one."""
        if not text:
            return 0.0
        return self.bits_many([text], condition=condition)[0]

    def cost_many(self, programs):
        """c(p) for several programs, batched. Unconditional, so they share no
        context and one forward pass can cover all of them."""
        if not programs:
            return []
        if self._empty is None:
            self._empty = self.bits("\n")
        todo = [p for p in programs if ("p", p) not in self._memo]
        if todo:
            got = self.bits_many([p + "\n" for p in todo])
            for p, v in zip(todo, got):
                self._memo[("p", p)] = v - self._empty
        return [self._memo[("p", p)] for p in programs]

    def cost(self, program):
        """c(p): the program priced on its own, no sequence given.

        This is the frame unquestionability needs. The terminating newline is
        charged so a program pays for stopping, and the empty program is priced
        the same way and subtracted, so c(p_empty) = 0 by construction.

        Unconditional, so it does not depend on the prefix length. That is what
        lets K and every rival be priced ONCE per item instead of once per
        prefix."""
        return self.cost_many([program])[0]

    def cost_given(self, program, shown):
        """c(p | x): the sequence first, the program after. Not used for
        unquestionability. Kept because the quantities in the plan's
        Section 'Quantities measured' include it and S2 needs it."""
        ctx = PRICE_TPL.format(terms=", ".join(shown))
        return (self.bits(" " + program + "\n", condition=ctx)
                - self.bits("\n", condition=ctx))


## 5 · The walk

One function, one item.
It returns the record and touches nothing else.


In [ ]:
def structural_rivals(terms, L):
    """The rivals that exist for every item at every prefix.

    Each must be a TOTAL program: it has to define a continuation for every
    later term, not just the next one. "Print the prefix, then a wrong term"
    is not a program for a sequence, it is a program for L+1 terms that then
    stops, so it cannot be a rival to a rule that runs forever. Every candidate
    here continues indefinitely by a trivial deterministic rule.

    Returns (kind, program, what it gives at L+1) for the ones that actually
    diverge. At least one always does unless x_{L+1} equals x_1, x_L and "0"
    at once, which the final fallback covers.
    """
    pre = ", ".join(terms[:L])
    nxt = terms[L]
    cands = [
        # cycle the prefix. Total, and the cheapest to state.
        ("cycle", f"{PRINT_TPL}{pre}, then repeat that list forever", terms[0]),
        # hold the last term. Total, and cheaper still on some machines.
        ("hold",  f"{PRINT_TPL}{pre}, then repeat {terms[L-1]} forever",
         terms[L-1]),
        # a constant unrelated to the item, so something always diverges
        ("const", f"{PRINT_TPL}{pre}, then 0 forever", "0"),
    ]
    out = [(k, p, g) for k, p, g in cands if g != nxt]
    if not out:
        # x_{L+1} is x_1, x_L and "0" at once, so the sequence is constant.
        # The repeat screen should have rejected such an item, but the walk must
        # not certify one by accident: a diverging total program always exists.
        alt = "1" if nxt != "1" else "2"
        out = [("const", f"{PRINT_TPL}{pre}, then {alt} forever", alt)]
    return out

def fit_beta(curve):
    """Least squares fit of cost = beta*L + c0 over the whole curve.

    beta is NOT needed to decide anything. At each certification prefix the
    structural rival is priced and compared with the budget directly, which is
    exact. The fit is only for saying how many terms an item WOULD need when no
    prefix in range clears the budget.

    It is fitted rather than taken from the endpoints because the cost does not
    grow monotonically: the strings are not nested, since the tail moves with L,
    so adding a term can make the whole string more predictable. On the first
    real run the cost fell at least once on 11 of 27 curves, and a two-point
    slope gave values from -0.1 to 17.8 bits per term. R2 says whether the line
    is worth extrapolating."""
    if len(curve) < 3:
        return dict(beta=None, c0=None, beta_r2=None)
    xs = [float(L) for L, _ in curve]
    ys = [c for _, c in curve]
    n = len(xs)
    mx, my = sum(xs)/n, sum(ys)/n
    sxx = sum((x-mx)**2 for x in xs)
    if sxx == 0:
        return dict(beta=None, c0=None, beta_r2=None)
    beta = sum((x-mx)*(y-my) for x, y in zip(xs, ys)) / sxx
    c0 = my - beta*mx
    ss_tot = sum((y-my)**2 for y in ys)
    ss_res = sum((y - (beta*x + c0))**2 for x, y in zip(xs, ys))
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0.0
    return dict(beta=round(beta, 2), c0=round(c0, 1), beta_r2=round(r2, 3),
                beta_scatter=round((ss_res/n)**0.5, 1))

def price_item(ref, item, progs, say=None):
    """Price one item under `ref`. Returns the record. No side effects.

    Everything is c(p), unconditional, so K and the composer rivals are priced
    once for the item and only the structural rivals change with L.

    Every prefix in walk_at is priced, and the walk does NOT stop at the first
    that certifies. Stopping there saved a few forward passes and cost two
    things: beta was fitted on however much of the ladder came before the stop,
    which left 7 of 22 items on the last run with no usable estimate at all,
    and the record could not answer what the item would do at a different
    delta, since the prefixes above n_M had never been priced.

    A verdict may still only be reached at a prefix the composer was actually
    asked for rivals at. The rest of the ladder is priced for beta and for the
    margins, and is marked can_certify = False.
    """
    say = say or (lambda *a: None)
    terms = item["terms"]

    equivs = [e["p"] for e in progs.get("equivalents", []) if e.get("verified")]
    _seed = seed_of(item)
    _bare = progs.get("bare")          # the shortest verified equivalent, by
                                       # CHARACTERS. K is the minimum in BITS,
                                       # and the two need not agree.
    if INCLUDE_AUTHORED_RULE and _seed and _seed not in equivs:
        equivs = [_seed] + equivs
    rivals = progs.get("rivals", [])

    # The item's own reach. A prefix above len(terms) - K_ADMIN leaves nothing
    # to ask a question with, and the generator sized every sequence by exactly
    # that rule, so the two agree by construction.
    l_max = len(terms) - K_ADMIN
    if L_MAX_CAP:
        l_max = min(l_max, L_MAX_CAP)
    l_max = min(l_max, len(terms) - 2)

    asked = sorted({r.get("diverges_at") for r in rivals
                    if r.get("diverges_at")} |
                   {L for L in (progs.get("asked_at") or []) if L})
    if not asked:
        asked = list(CERTIFY_AT_FALLBACK)
    certify_at = [L for L in asked if L0 <= L <= l_max]

    # beta needs its points spread over the range the item can actually reach,
    # not over a window that stops at 20.
    if l_max > L0 and BETA_LADDER_N >= 2:
        _step = (l_max - L0) / (BETA_LADDER_N - 1)
        _ladder = sorted({int(round(L0 + i*_step)) for i in range(BETA_LADDER_N)})
    else:
        _ladder = [L0]
    walk_at = sorted({L for L in (_ladder + certify_at) if L0 <= L <= l_max})

    if not certify_at:
        return dict(id=item["id"],
                    skipped=(f"rivals were asked for at {asked}, none of which "
                             f"fits inside {len(terms)} terms with {K_ADMIN} "
                             f"left to ask with"),
                    no_tested_prefix=True)
    if not equivs:
        return dict(id=item["id"], skipped="no program reproduces the sequence",
                    no_equivalents=True)

    # ---- priced once for the whole item -----------------------------------
    n_auth = 1 if (INCLUDE_AUTHORED_RULE and _seed in equivs) else 0
    say(f"      pricing {len(equivs)} candidates for K, once, in batches of "
        f"{BATCH_ROWS}: {len(equivs)-n_auth} verified equivalents"
        + (" plus the authored rule" if n_auth else "")
        + ". The structural rivals are priced inside the walk, since they "
          "change with L.")
    _t = time.time()
    priced = list(zip(equivs, ref.cost_many(equivs)))
    say(f"      done in {time.time()-_t:.1f}s")
    p_star, K = min(priced, key=lambda c: c[1])
    budget = K + DELTA
    src = "the seed rule" if p_star == _seed else "an equivalent"
    say(f"      K = {K:.1f} bits from {src}; dearest {max(c[1] for c in priced):.1f}. "
        f"budget = K + delta = {budget:.1f}")

    # Does the shortest program in CHARACTERS also win in BITS? The generator's
    # term ladder rests on a proxy of about 2.2 bits per character, and this is
    # the free test of it: everything here is priced anyway.
    bare_bits = next((c for p, c in priced if p == _bare), None)
    bare_is_star = (_bare is not None and p_star == _bare)
    if _bare is not None and bare_bits is not None:
        say(f"      bare (shortest in chars, {len(_bare)}) prices at "
            f"{bare_bits:.1f} bits; it "
            + ("IS" if bare_is_star else
               f"is NOT the cheapest, which costs {K:.1f}")
            + " the argmin in bits")

    # Rivals are only PRICED here. That they reproduce the prefix and diverge
    # was established by execution at generation time and is taken as given.
    at_L = collections.defaultdict(list)
    in_range = [r for r in rivals
                if r.get("diverges_at") is not None
                and r["diverges_at"] in certify_at]
    _out_of = [r for r in rivals
               if r.get("diverges_at") is not None
               and r["diverges_at"] not in certify_at]
    if _out_of:
        say(f"      {len(_out_of)} rival(s) sit at prefixes outside CERTIFY_AT "
            f"({sorted({r['diverges_at'] for r in _out_of})}) and are not priced: "
            f"no verdict is reached there, so their cost would change nothing.")
    if in_range:
        say(f"      pricing {len(in_range)} composer rivals, once "
            f"(divergence is taken as given, not re-checked)")
        _t = time.time()
        costs = ref.cost_many([r["p"] for r in in_range])
        say(f"      done in {time.time()-_t:.1f}s")
        for r, c_q in zip(in_range, costs):
            at_L[r["diverges_at"]].append(
                dict(p=r["p"], cost=c_q, gives=r.get("gives"),
                     inside=c_q <= budget))
    for d in sorted(at_L):
        n_in = sum(1 for r in at_L[d] if r["inside"])
        say(f"      L={d}: {len(at_L[d])} composer rival(s), {n_in} inside budget"
            + (f", cheapest {min(r['cost'] for r in at_L[d]):.1f}" if at_L[d] else ""))

    rec = dict(id=item["id"], machine=REF_MACHINE, machine_tag=TAG,
               delta=DELTA,
               # how delta was arrived at, so a verdict can never be read
               # without knowing the budget it was reached under
               delta_source=(DELTA_SOURCE if "DELTA_SOURCE" in globals()
                             else "unknown"),
               l0=L0, l_max=l_max, frame="c(p) unconditional",
               certify_at=certify_at, walk_at=walk_at,
               n_equivalents=len(equivs), n_rivals=len(rivals),
               K=round(K, 1), p_star=p_star, K_source=src,
               bare=_bare,
               bare_bits=(round(bare_bits, 1) if bare_bits is not None else None),
               bare_is_argmin=bare_is_star,
               bare_gap_bits=(round(bare_bits - K, 1)
                              if bare_bits is not None else None),
               budget=round(budget, 1),
               composer_rivals_at=sorted(at_L), per_L={}, blocked=[],
               # prefix -> bits of headroom, at the verdict prefixes AND the
               # beta-ladder ones. delta never enters here.
               margins={},
               priced_at=datetime.datetime.now().isoformat(timespec="minutes"))

    # ---- the walk ---------------------------------------------------------
    # Every prefix in walk_at is priced, because beta needs the curve. Only a
    # prefix in certify_at can end the walk, because only there was a search
    # ever made. A prefix priced for beta alone is recorded with
    # can_certify = False so the record says why it did not stop there.
    n_m, struct_curve = None, []
    for L in walk_at:
        _cands = structural_rivals(terms, L)
        if not STRUCT_PRICE_ALL:
            _cands = _cands[:1]     # they cost about the same; take the first
        struct = list(zip([c[0] for c in _cands], [c[1] for c in _cands],
                          [c[2] for c in _cands],
                          ref.cost_many([c[1] for c in _cands])))
        cheapest = min(struct, key=lambda x: x[3])
        struct_curve.append((L, cheapest[3]))
        can_certify = L in certify_at
        blockers = [dict(p=k, cost=round(c, 1), kind="structural", gives=g)
                    for k, _, g, c in struct if c <= budget]
        blockers += [dict(p=r["p"][:120], cost=round(r["cost"], 1),
                          kind="composer", gives=r["gives"])
                     for r in at_L.get(L, []) if r["inside"]]
        _cheapest_rival = min([cheapest[3]]
                              + [r["cost"] for r in at_L.get(L, [])])
        say(f"      L={L}: cheapest structural rival ({cheapest[0]}) "
            f"{cheapest[3]:.1f} vs budget {budget:.1f}"
            + (f"  -> {len(blockers)} blocker(s)" if blockers
               else "  -> nothing blocks")
            + ("" if can_certify else "   [beta only, no verdict here]"))
        rec["per_L"][str(L)] = dict(
            struct_cheapest=round(cheapest[3], 1), struct_kind=cheapest[0],
            struct_all={k: round(c, 1) for k, _, _, c in struct},
            slack=round(cheapest[3] - budget, 1),
            n_composer_rivals=len(at_L.get(L, [])),
            n_blockers=len(blockers), blockers=blockers,
            can_certify=can_certify,
            # The headroom at this prefix, in bits: the cheapest rival of any
            # kind, less K. This is the whole verdict with delta divided out.
            # An item certifies at L exactly when margin > delta, so storing it
            # makes n_M a function of delta that can be evaluated afterwards for
            # any value, on any machine, with no pricing at all.
            margin=round(_cheapest_rival - K, 1))
        rec["margins"][str(L)] = round(_cheapest_rival - K, 1)
        if can_certify:
            if not blockers and n_m is None:
                n_m = L
            elif blockers:
                rec["blocked"].append(L)

    rec["n_M"] = n_m
    rec["certified"] = n_m is not None
    rec["struct_curve"] = [(L, round(c, 1)) for L, c in struct_curve]
    rec.update(fit_beta(struct_curve))
    if rec.get("beta") is not None:
        say(f"      beta = {rec['beta']} bits per term, c0 = {rec['c0']}, "
            f"R2 = {rec['beta_r2']} (scatter {rec['beta_scatter']} bits)")
    if n_m is not None:
        rec["slack"] = rec["per_L"][str(n_m)]["slack"]
        rec["tested_by_composer"] = (n_m in at_L)
    elif EXTRAPOLATE_L and rec.get("beta", 0) > 0:
        # Extrapolate from the FITTED line, not from the last point, and only
        # when the fit is good enough to be worth extrapolating.
        need = (budget - rec["c0"]) / rec["beta"]
        if rec["beta_r2"] >= BETA_MIN_R2 and need > rec["l_max"]:
            rec["L_needed_estimate"] = int(math.ceil(need))
            rec["L_needed_quality"] = "fitted"
        else:
            rec["L_needed_quality"] = (
                f"no estimate: the cost of the structural rival does not grow "
                f"cleanly with L (R2 = {rec['beta_r2']:.2f})")
        rec["gen_terms_needed_estimate"] = progs.get("terms_needed_estimate")
        rec["gen_terms_available"] = progs.get("terms_available")
    return rec

def n_M_at(rec, delta):
    """n_M for any delta, from the stored margins. No pricing.

    An item certifies at L exactly when the cheapest rival there costs more than
    K + delta, that is when margin(L) > delta. Since the margins are recorded
    with delta divided out, every question of the form "what would this battery
    look like at a different tolerance" is arithmetic on a file.

    That matters because delta is not free to choose. Definition 4.1 bounds it
    below the cheapest exception clause, and the margin under that bound is a
    judgement. With the margins stored, the whole curve can be reported instead
    of one point on it, and changing the tolerance costs nothing rather than a
    GPU run, so there is no incentive to leave it unchecked.
    """
    ok = [int(L) for L, m in rec.get("margins", {}).items()
          if m > delta and rec.get("per_L", {}).get(L, {}).get("can_certify")]
    return min(ok) if ok else None

def verdict(rec):
    """One short line, and whether it earns a mark."""
    if rec.get("skipped"):
        return f"skipped: {rec['skipped']}", None
    if rec["certified"]:
        s = (f"n_M = {rec['n_M']}, K = {rec['K']:.0f} bits, "
             f"slack {rec['slack']:+.0f}")
        if not rec.get("tested_by_composer"):
            s += "  [no composer rival at that prefix]"
        return s, None
    kinds = {b["kind"] for L in rec["blocked"] for b in rec["per_L"][str(L)]["blockers"]}
    est = rec.get("L_needed_estimate")
    s = (f"questionable at every tested prefix {rec['certify_at']}, blocked by "
         + ("the structural rival alone" if kinds == {"structural"}
            else "composer rivals" if kinds == {"composer"} else "both"))
    if est:
        s += f"; would need about {est} terms at {rec.get('beta')} bits per term"
    return s, ("structural" if kinds == {"structural"} else "blocked")


## 6 · Load the reference machine

Separate from the run because loading takes minutes and a session may want to
price several batches.
If a machine of the same name is already loaded in this
kernel, it is reused and nothing is downloaded again.


In [ ]:
# Reuse a machine already loaded in this kernel. Loading an 8B model takes
# several minutes, and there is no reason to pay that twice in one session.
_want = REF_MACHINE
_why = None
try:
    if REF.name != _want:
        _why = f"a different machine is loaded ({REF.name})"
    elif getattr(REF, "build", None) != BUILD:
        _why = (f"the loaded machine was built by "
                f"{getattr(REF, 'build', 'an unknown version')}, not {BUILD}")
    elif not all(hasattr(REF, m) for m in ("cost", "bits", "cost_given")):
        _why = "the loaded machine is missing methods this version needs"
except NameError:
    _why = "nothing is loaded in this kernel"

if _why is None:
    log(f"  {_want} is already loaded on {REF.dev}, reusing it "
        f"({REF.calls} forward passes so far)")
else:
    log(f"  loading: {_why}")
    try:
        del REF
    except NameError:
        pass
    REF = Machine(_want, four_bit=LOAD_4BIT)


## 6b · The rival budget, $\delta$

$\delta$ is the only parameter in Definition 4.1, and it is bounded rather than
chosen.
A rival must fit inside $c_\mathcal{M}(p^\star) + \delta$, and one
rival is always available: take $p^\star$ and bolt an **exception clause** onto
it, overriding the term at $L+1$.
If $\delta$ reached the cost of that clause,
the clause would sit inside the window at every prefix and nothing could ever
certify.
So

$$\delta \;<\; c_\mathcal{M}(\text{cheapest exception clause}).$$

The paper measured that clause at upwards of 40 bits and used $\delta = 15$,
which is 25 bits of headroom.
The bound is machine-relative, so a new reference
machine needs the clause repriced before the inherited 15 can be trusted.

The **margin** is ours to choose; the **bound** is not.
A larger margin
certifies fewer items and each verdict then survives a wider search later, which
is the right way to compensate for a rival search that is necessarily
incomplete.

This depends only on the reference machine, not on which items a run prices, so
it is measured once and cached in the pricing file.
Every record carries
`delta_source`, and the value is printed loudly enough to interrupt a run over.


In [ ]:
# ---------------------------------------------------------------------------
# delta is NOT free. Definition 4.1 fixes what it may be:
#
#   "The tolerance delta is the only parameter: it must be smaller than the
#    cost of the cheapest exception clause, since otherwise 'p*, but x'_{L+1}
#    at term L+1' would be a rival inside the window at every length and
#    nothing could certify. Under the reference prior such a clause costs
#    upwards of forty bits; we use delta = 15."
#
# So the bound is delta < c_M(cheapest exception clause), and 15 was chosen
# with 25 bits of headroom under a measured 40. That 40 was measured under an
# EARLIER machine. The bound is machine-relative, so on a new reference machine
# the clause has to be repriced before the inherited 15 can be trusted.
#
# The margin is the part that is ours to choose. It compensates for our rival
# search being incomplete: a smaller delta certifies fewer items but each
# verdict survives a wider search later. The bound is not ours to choose.
#
# This depends only on the reference machine, never on which items a run
# happens to price, so it is measured once and cached in the pricing file.
# `delta_source` records which way the value came, and it is printed loudly
# enough to stop a run over.

DELTA_MODE     = "derived"   # "explicit" uses DELTA_EXPLICIT unchanged.
                             # "derived"  measures the exception clause under
                             #            this machine and subtracts the margin.
DELTA_EXPLICIT = 15.0        # the inherited value, from the paper
# named here, not in the calibration cell below: delta is read from this
# file long before that cell runs
CALIBRATION      = "ctest_calibration.json"

DELTA_MARGIN = 0.0        # bits subtracted from the clause. See DELTA_KEEP.
DELTA_KEEP   = 1.0
DELTA_REMEASURE = False      # True re-measures even when a cached value exists
DELTA_PROBE_ITEMS = 6        # items sampled for the clause. The bound is the
                             # CHEAPEST clause, so more probes can only lower it.

def measure_exception_clause(ref, items, progs, n_items=6, say=log):
    """Price the cheapest exception clause under `ref`.

    The clause is what turns a correct program into a rival: keep p* and
    override the term at L+1. Its cost is the INCREMENT over p*, since the
    rival is "p* plus the clause" and has to fit inside c(p*) + delta. So the
    clause is inside the window exactly when its cost is at most delta, which
    is why delta must be below it.

    Several phrasings are tried on several items and the minimum is taken,
    because the bound is against the cheapest clause anyone could write, not
    against a typical one.
    """
    probes, seen = [], 0
    for b in items:
        if seen >= n_items:
            break
        pr = progs.get(b["id"], {})
        p = pr.get("bare") or seed_of(b)
        terms = b.get("terms") or []
        if not p or len(terms) < L0 + 2:
            continue
        seen += 1
        L = L0
        true_next = terms[L]
        wrong = "0" if true_next != "0" else "1"
        for tpl in (
            "{p}, except term {n} is {w}",
            "{p}, but output {w} at term {n}",
            "{p}. At term {n} output {w} instead.",
            "{p}, except the {n}th term, which is {w}",
        ):
            probes.append((b["id"], p,
                           tpl.format(p=p, n=L + 1, w=wrong)))
    if not probes:
        return None, []

    base = ref.cost_many([p for _, p, _ in probes])
    full = ref.cost_many([q for _, _, q in probes])
    rows = []
    for (iid, p, q), cb, cf in zip(probes, base, full):
        rows.append(dict(item=iid, clause_bits=round(cf - cb, 1),
                         base_bits=round(cb, 1), text=q[-60:]))
    rows.sort(key=lambda r: r["clause_bits"])
    return rows[0]["clause_bits"], rows

stage("THE RIVAL BUDGET, delta")

_ref = None
try:
    _cp = find(CALIBRATION) or CALIBRATION
    _ref = (json.load(open(_cp)).get("machines", {})
            .get(REF_MACHINE, {}).get("reference"))
except Exception:
    _ref = None

_cached = PRICED.get("_delta_calibration")
_reuse = (_cached and not DELTA_REMEASURE
          and _cached.get("machine") == REF_MACHINE
          and _cached.get("margin") == DELTA_MARGIN)

if DELTA_MODE != "explicit" and _ref and _ref.get("clause_bits") and not DELTA_REMEASURE:
    _clause = float(_ref["clause_bits"])
    DELTA = round(min(_clause - DELTA_MARGIN, _clause * DELTA_KEEP), 1)
    DELTA_SOURCE = f"calibrator, probes {_ref.get('probes','?')}"
    log(f"  from {CALIBRATION}, measured on fixed probes:")
    log(f"    cheapest exception clause  {_clause} bits"
        f"   ({_ref.get('n_probes','?')} probes, {_ref.get('measured_on','?')})")
    log(f"    margin                     {DELTA_MARGIN} bits, or keep "
        f"{DELTA_KEEP:.0%}, whichever is smaller")
    log(f"    delta                      {DELTA}")
    if DELTA <= 0:
        DELTA = round(_clause * DELTA_KEEP, 1)
        log(f"  the fixed margin does not fit under this clause, so the "
            f"fraction is used: {DELTA}")
    if _cached and _cached.get("clause_bits"):
        log(f"  ({PRICING} holds an older measurement of "
            f"{_cached['clause_bits']} bits, taken on battery items. It is not "
            f"used.)")
elif DELTA_MODE == "explicit":
    DELTA = float(DELTA_EXPLICIT)
    DELTA_SOURCE = "explicit"
    log(f"  DELTA_MODE is 'explicit', so delta = {DELTA} bits as given.")
    log(f"  Definition 4.1 requires delta < c_M(cheapest exception clause).")
    log(f"  Nothing here checks that under {TAG}. Set DELTA_MODE = 'derived'")
    log(f"  to measure it.")
    _clause = _cached.get("clause_bits") if _cached else None
    if _clause is not None:
        log(f"  A cached measurement exists: the clause priced at {_clause} bits"
            f" under this machine.")
        if DELTA >= _clause:
            log(f"  WARNING: delta = {DELTA} is NOT below that bound. Every item"
                f" has a rival inside the window at every prefix and nothing can"
                f" certify.")
elif _reuse:
    DELTA = float(_cached["delta"])
    DELTA_SOURCE = "derived, cached"
    log(f"  reusing the measurement already in {PRICING}, since delta depends")
    log(f"  on the reference machine and not on which items a run prices.")
    log(f"    cheapest exception clause  {_cached['clause_bits']} bits"
        f"   (measured {_cached.get('measured_on','?')})")
    log(f"    margin                     {DELTA_MARGIN}")
    log(f"    delta                      {DELTA}")
    log(f"  Set DELTA_REMEASURE = True to measure it again.")
else:
    log(f"  measuring the cheapest exception clause under {REF_MACHINE}.")
    log(f"  This is a property of the machine, so it is done once and cached.")
    _t = time.time()
    _clause, _rows = measure_exception_clause(
        REF, ITEMS, PR, n_items=DELTA_PROBE_ITEMS)
    if _clause is None:
        DELTA = float(DELTA_EXPLICIT)
        DELTA_SOURCE = "explicit, no probe possible"
        log(f"  no item could be probed, so delta falls back to {DELTA}")
    else:
        DELTA = round(_clause - DELTA_MARGIN, 1)
        DELTA_SOURCE = "derived"
        log(f"  measured in {time.time()-_t:.0f}s over {len(_rows)} probes")
        log(f"\n    {'item':10s} {'clause':>7s}  text")
        for r in _rows[:5]:
            log(f"    {r['item']:10s} {r['clause_bits']:7.1f}  ...{r['text']}")
        if len(_rows) > 5:
            log(f"    {'':10s} {'':7s}  ... and {len(_rows)-5} more, "
                f"dearest {_rows[-1]['clause_bits']:.1f}")
        log(f"\n    cheapest exception clause  {_clause:.1f} bits   <- the bound")
        log(f"    margin                     {DELTA_MARGIN}")
        log(f"    delta                      {DELTA}")
        PRICED["_delta_calibration"] = dict(
            machine=REF_MACHINE, machine_tag=TAG, clause_bits=_clause,
            margin=DELTA_MARGIN, delta=DELTA, n_probes=len(_rows),
            measured_by=BUILD,
            measured_on=datetime.datetime.now().isoformat(timespec="minutes"),
            probes=_rows[:12])
        save()

# ---- say plainly how far this is from the published value -----------------
_pub = 15.0
log(f"\n  delta = {DELTA} bits   ({DELTA_SOURCE})")
if DELTA <= 0:
    log(f"  STOP. delta is not positive, so no rival can ever be inside the")
    log(f"  window and every item certifies trivially. Either the clause priced")
    log(f"  far below the paper's 40 bits under this machine, or the margin of")
    log(f"  {DELTA_MARGIN} is too large for it.")
elif abs(DELTA - _pub) <= 3:
    log(f"  That is within 3 bits of the {_pub} used in the paper, so the")
    log(f"  inherited value carries over to this machine.")
else:
    log(f"  The paper uses {_pub}. This differs by {DELTA-_pub:+.1f} bits, so the")
    log(f"  exception clause is priced differently by {TAG} than by the machine")
    log(f"  the paper used. That is a finding about the machine and it belongs")
    log(f"  in the write-up. Interrupt now if you would rather run at {_pub}:")
    log(f"  set DELTA_MODE = 'explicit' and re-run this cell.")
log(f"\n  A rival blocks when c_M(q) <= K + {DELTA}. Nothing below this cell")
log(f"  reads DELTA_EXPLICIT, so the value above is the one the walk uses.")


## 7 · Run

One item at a time, saved after each.
Each item announces itself and its
programs **before** any pricing starts, then reports the walk prefix by prefix:
the cheapest program and where it came from, the listing rival under both
baselines, and every composer rival with its cost and whether it falls inside
the window.

`VERBOSE = False` collapses that to one line per item.


In [ ]:
VERBOSE = True

T0 = time.time()
stage(f"PRICING UNDER {REF_MACHINE}")

targets = []
for b in ITEMS:
    if PRICE_PREFIX and not b["id"].startswith(PRICE_PREFIX): continue
    if PRICE_ONLY and b["id"] not in PRICE_ONLY: continue
    if SKIP_EXCLUDED and not usable(b): continue
    _stale = b["id"] in DONE and not DONE[b["id"]].get("margins") \
             and not DONE[b["id"]].get("skipped")
    if b["id"] in DONE and not REPRICE and not _stale: continue
    if b["id"] not in PR: continue
    targets.append(b)
if PRICE_ONLY:
    # keep the order given in the config, so a run cut short still answers the
    # first question asked of it
    _order = {i: n for n, i in enumerate(PRICE_ONLY)}
    targets.sort(key=lambda b: _order.get(b["id"], 10**6))
if MAX_ITEMS: targets = targets[:MAX_ITEMS]

if PRICE_ONLY:
    _missing = [i for i in PRICE_ONLY
                if i not in {b["id"] for b in targets}
                and i not in DONE]
    if _missing:
        log(f"  {len(_missing)} named item(s) are not in the queue: "
            f"{', '.join(_missing)}")
        log(f"  (already priced, marked, or absent from the programs file)")

def _gen_est(iid):
    """What the generator predicted, before anything was priced. `fits` means
    it thinks the terms suffice; this run is the first test of that."""
    p = PR.get(iid, {})
    need, avail = p.get("terms_needed_estimate"), p.get("terms_available")
    if need is None or avail is None:
        return None, None, "?"
    return need, avail, ("fits" if need <= avail else f"short by {need-avail}")

_old_fmt = sum(1 for b in targets if b["id"] in DONE)
if _old_fmt:
    log(f"  {_old_fmt} item(s) already have a record but no margins, so they "
        f"predate 1.6.0 and are being re-priced. Their beta was fitted on a "
        f"ladder the old early stop cut short.")
log(f"  {len(targets)} items to price"
    + (f", capped at {MAX_ITEMS}" if MAX_ITEMS else ""))

if not targets:
    _match = [b for b in ITEMS
              if not PRICE_PREFIX or b["id"].startswith(PRICE_PREFIX)]
    _done = sum(1 for b in _match if b["id"] in DONE)
    _noprog = sum(1 for b in _match if b["id"] not in PR)
    _excl = sum(1 for b in _match if SKIP_EXCLUDED and not usable(b))
    log(f"\n  NOTHING TO DO, and here is why.")
    log(f"    PRICE_PREFIX is {PRICE_PREFIX!r}, which matches {len(_match)} of "
        f"{len(ITEMS)} items in the battery.")
    if _done:
        log(f"    {_done} of those are already priced. REPRICE is {REPRICE}, so "
            f"they are skipped.")
    if _noprog:
        log(f"    {_noprog} have no programs record and cannot be priced.")
    if _excl:
        log(f"    {_excl} are excluded.")
    _pref = sorted({b["id"].split("-")[0] for b in ITEMS})
    _fresh = {p: sum(1 for b in ITEMS
                     if b["id"].startswith(p) and b["id"] not in DONE
                     and b["id"] in PR)
              for p in _pref}
    log(f"    unpriced items by batch: "
        + ", ".join(f"{p} {n}" for p, n in sorted(_fresh.items()) if n)
        + ("  (none anywhere)" if not any(_fresh.values()) else ""))
    log(f"    Set PRICE_PREFIX to one of those, or clear it for all of them.")
    log(f"    The summary below describes the file as it stands, not this run.")
if targets:
    log(f"\n  THE QUEUE:")
    log(f"    {'#':>3s}  {'item':10s} {'cell':7s} {'terms':>5s} {'equiv':>5s} "
        f"{'rivals':>6s}  {'generator estimate':18s}  rivals sit at L")
    for n, b in enumerate(targets, 1):
        pr = PR.get(b["id"], {})
        nv = sum(1 for e in pr.get("equivalents", []) if e.get("verified"))
        rv = pr.get("rivals", [])
        at = sorted({r.get("diverges_at") for r in rv if r.get("diverges_at")})
        need, avail, verdict_txt = _gen_est(b["id"])
        est = (f"needs {need}/{avail} {verdict_txt}" if need is not None else "-")
        log(f"    {n:3d}  {b['id']:10s} "
            f"r{b.get('want_retrieval')}t{b.get('want_transformation'):<4d} "
            f"{len(b['terms']):5d} {nv:5d} {len(rv):6d}  {est:18s}  "
            f"{at if at else 'no composer rival; the structural ones still apply'}")
    log(f"\n  Verdicts are reached only where the composer was asked for "
        f"rivals, which is now read per item. Other prefixes are priced for "
        f"beta and cannot end the walk.")

marked = collections.Counter()
for i, b in enumerate(targets):
    # `and TIME_BUDGET_MIN` first, or 0 means "stop at once" rather than "no
    # limit": the check is elapsed > budget and any elapsed time exceeds zero.
    # A run set to 0 for no limit priced nothing, reporting "time budget of 0
    # min reached" before its first item. The generator already had this guard.
    if TIME_BUDGET_MIN and (time.time() - T0) / 60 > TIME_BUDGET_MIN:
        log(f"\n  time budget of {TIME_BUDGET_MIN} min reached, stopping cleanly")
        break
    pr = PR[b["id"]]
    nv = sum(1 for e in pr.get("equivalents", []) if e.get("verified"))
    rv = pr.get("rivals", [])
    at = sorted({r.get("diverges_at") for r in rv if r.get("diverges_at")})
    need, avail, est_txt = _gen_est(b["id"])

    # announced BEFORE the work, so a slow item is identifiable while it runs
    log(f"\n  {'-'*68}")
    log(f"  [{i+1}/{len(targets)}] {b['id']}  "
        f"r{b.get('want_retrieval')}t{b.get('want_transformation')}  "
        f"{len(targets)-i-1} left after this one")
    log(f"      THEME {b.get('theme','')}  ({b.get('domain_real','')})")
    log(f"      {len(b['terms'])} terms; {nv} verified equivalents; "
        f"{len(rv)} composer rivals at L={at if at else 'nowhere'}")
    if need is not None:
        log(f"      generator estimate: needs {need} terms of {avail} -> {est_txt}")
    log(f"      shown at L={L0}: {', '.join(b['terms'][:L0])}"
        f"   then {b['terms'][L0]!r}")

    t1, calls0 = time.time(), REF.calls
    rec = price_item(REF, b, pr, say=(log if VERBOSE else None))
    DONE[b["id"]] = rec

    if WRITE_TO_BATTERY and not rec.get("skipped"):
        b[f"n_M_{TAG}"] = rec["n_M"]
        b[f"K_{TAG}"] = rec.get("K")
        b[f"slack_{TAG}"] = rec.get("slack")
        if rec.get("L_needed_estimate"):
            b[f"L_needed_{TAG}"] = rec["L_needed_estimate"]
        if rec["certified"] and not rec.get("tested_by_composer"):
            b[f"struct_only_{TAG}"] = True

    line, why = verdict(rec)
    if why and MARK_QUESTIONABLE:
        note = (f"{why}: under {TAG} at delta={DELTA}, c(p) unconditional, a "
                f"rival within delta survives at every tested prefix "
                f"{rec.get('certify_at')}"
                + (f"; estimated {rec['L_needed_estimate']} terms needed"
                   if rec.get("L_needed_estimate") else ""))
        if mark(b, MARK_VALUE, note):
            marked[why] += 1
            line += f"  -> marked {MARK_VALUE}"
    save()
    left = len(targets) - (i + 1)
    per = (time.time() - T0) / (i + 1)
    eta = datetime.datetime.now() + datetime.timedelta(seconds=per * left)
    _n = REF.calls - calls0
    _dt = time.time() - t1
    log(f"      VERDICT  {line}")
    log(f"      {_dt:.0f}s, {_n} programs priced"
        + (f" ({_dt/_n:.2f}s each)" if _n else "") + ". "
        + (f"{left} left, about {per*left/60:.0f} min (~{eta:%H:%M}), next is "
           f"{targets[i+1]['id']}" if left else "queue finished"))
    if STATE_PUSH_EVERY and (i + 1) % STATE_PUSH_EVERY == 0:
        state_save(f"{BUILD}: {i+1} items priced under {TAG}")


## 8 · Calibration for the generator

The constants that say what this reference machine charges, written here and
read by the generator.
This notebook has the GPU and has just measured all of
them; the generator has neither, and that separation is the point.

The file did not exist until now, so the generator has been running on its
built-in defaults for four batteries.
The ladder that sizes every sequence rests
on those constants, which is why this matters more than it looks.

`bits_per_char_p80` is the one to notice.
The median says what the population
does; the p80 is what a ladder should size by, because the two errors are not
symmetric.
Too many terms costs a longer sequence and nothing else, since
$n_\mathcal{M}$ is the shortest certifying prefix and the rest is headroom.
Too
few cannot be repaired without regenerating the item.


In [ ]:
# ---------------------------------------------------------------------------
# ctest_calibration.json
#
# The constants that describe what THIS reference machine charges, written here
# because this is the notebook with the GPU and it has just measured all of
# them. The generator reads the file and never writes it, which is what keeps
# the generator machine-independent and free of GPU work.
#
# Until now the file did not exist and the generator ran on its built-in
# defaults for four batteries. Two of those defaults were guesses that the runs
# have since contradicted, and the ladder that sizes every sequence rests on
# them, so this closes the loop.
#
# Entries are filled where missing and NOT overwritten. Re-measuring needs
# CALIB_REMEASURE, because otherwise the constants drift with whichever items a
# run happened to price and a generation run from last week stops being
# reproducible.

CALIB_REMEASURE  = False
CALIB_MIN_ITEMS  = 10      # below this a run says so and writes nothing

# This runs BEFORE the summary, and that ordering matters. The summary cell
# ends with the state push, and a push REPLACES the dataset with whatever is
# staged at that moment. Written afterwards, the file sat in the working
# directory and reached the dataset only if someone noticed and copied it by
# hand. STATE_PATTERNS matches ctest_*.json, so writing it here means it is
# staged with everything else.
stage("CALIBRATION FOR THE GENERATOR")

_done = [r for r in DONE.values() if not r.get("skipped")]
_bpc = [r["K"] / len(r["bare"]) for r in _done
        if r.get("K") and r.get("bare")]
_beta = [r["beta"] for r in _done
         if r.get("beta") and (r.get("beta_r2") or 0) >= BETA_MIN_R2]
_c0 = [r["c0"] for r in _done
       if r.get("c0") is not None and (r.get("beta_r2") or 0) >= BETA_MIN_R2]
_dc = PRICED.get("_delta_calibration") or {}

# Report the fit WHETHER OR NOT it is written. A run that falls below the bar
# still measured something, and the numbers are the only evidence about whether
# these constants describe the MACHINE or the BATTERY they were fitted on. beta
# and bits_per_char are properties of both: beta is how compressible these
# items' terms are, bits_per_char how densely their rules are written. So the
# thing to watch across batteries is whether they move. Two earlier fits gave
# 2.08 and 2.13 bits per character, 4.32 and 4.23 beta, on different items.
def _q(v, p):
    v = sorted(v)
    return v[min(len(v) - 1, int(p * len(v)))] if v else None

log(f"\n  WHAT THIS RUN MEASURED, written or not:")
log(f"    items priced            {len(_done)}   (need {CALIB_MIN_ITEMS})")
if _bpc:
    log(f"    bits per character      median {st.median(_bpc):.2f}, "
        f"p80 {_q(_bpc, 0.80):.2f}, range {min(_bpc):.2f} to {max(_bpc):.2f}"
        f"   (n = {len(_bpc)})")
else:
    log(f"    bits per character      no item had both a K and a bare rule")
if _beta:
    log(f"    beta                    median {st.median(_beta):.2f}, "
        f"p80 {_q(_beta, 0.80):.2f}, range {min(_beta):.2f} to {max(_beta):.2f}"
        f"   (n = {len(_beta)} fits at R2 >= {BETA_MIN_R2})")
else:
    _tried = sum(1 for r in _done if r.get("beta") is not None)
    log(f"    beta                    no fit cleared R2 >= {BETA_MIN_R2} "
        f"({_tried} were attempted)")
if _c0:
    log(f"    c0                      median {st.median(_c0):.1f}   "
        f"(n = {len(_c0)})")
if _dc:
    log(f"    exception clause        {_dc.get('clause_bits')} bits, "
        f"delta {_dc.get('delta')} at margin {_dc.get('margin')}")
log(f"    Compare these against an earlier entry. If they barely move across "
    f"batteries, the constants describe the machine; if they swing, they")
log(f"    describe whichever items were priced, and the plan should say so.")

if len(_done) < CALIB_MIN_ITEMS:
    log(f"\n  NOT WRITTEN: only {len(_done)} items priced, below "
        f"CALIB_MIN_ITEMS = {CALIB_MIN_ITEMS}. A constant fitted on a handful "
        f"of items would be worse than the default it replaced.")
    log(f"  The generator will fall back to its built-in defaults and say so.")
elif not _bpc or not _beta:
    log(f"\n  NOT WRITTEN: {len(_bpc)} cost fits and {len(_beta)} beta fits. "
        f"The numbers above still stand as a measurement; they are simply too "
        f"thin to hand to the generator.")
else:
    _bpc.sort()
    def _pct(v, p): return v[min(len(v) - 1, int(p * len(v)))]

    entry = dict(
        machine=REF_MACHINE, machine_tag=TAG,
        bits_per_char=round(st.median(_bpc), 2),
        bits_per_char_p80=round(_pct(_bpc, 0.80), 2),
        bits_per_char_range=[round(_bpc[0], 2), round(_bpc[-1], 2)],
        # beta: the central value sets the prompt budget, the high one decides
        # when to stop spending on an item.
        beta_listing=round(st.median(_beta), 2),
        beta_gate=round(_pct(sorted(_beta), 0.80), 2),
        c0_listing=round(st.median(_c0), 1) if _c0 else None,
        clause_bits=_dc.get("clause_bits"),
        delta=_dc.get("delta"),
        delta_margin=_dc.get("margin"),
        n_items=len(_done), n_beta_fits=len(_beta),
        measured_by=BUILD,
        measured_on=datetime.datetime.now().isoformat(timespec="minutes"))

    _path = find(CALIBRATION) or CALIBRATION
    try:
        _all = json.load(open(_path))
    except Exception:
        _all = {"_README": ("Constants describing each reference machine. "
                            "Written by the unquestionability notebook, which "
                            "has the GPU; read by the generator, which does "
                            "not. Filled where missing, never overwritten "
                            "silently."),
                "machines": {}}
    _all.setdefault("machines", {})
    _old = _all["machines"].get(REF_MACHINE)

    if _old and not CALIB_REMEASURE:
        log(f"  {REF_MACHINE} already has an entry, measured "
            f"{_old.get('measured_on','?')} on {_old.get('n_items','?')} items. "
            f"Left alone.")
        log(f"  This run would have written:")
        for k in ("bits_per_char", "bits_per_char_p80", "beta_listing",
                  "c0_listing"):
            log(f"    {k:20s} {_old.get(k)}  ->  {entry[k]}")
        log(f"  Set CALIB_REMEASURE = True to replace it.")
    else:
        _all["machines"][REF_MACHINE] = entry
        with open(CALIBRATION, "w") as f:
            json.dump(_all, f, indent=1)
        log(f"  wrote {CALIBRATION} for {REF_MACHINE}, from {len(_done)} items")
        for k in ("bits_per_char", "bits_per_char_p80", "bits_per_char_range",
                  "beta_listing", "beta_gate", "c0_listing", "clause_bits",
                  "delta"):
            log(f"    {k:22s} {entry[k]}")
        log(f"\n  The generator reads this on its next run. bits_per_char_p80 "
            f"is the one that sizes sequences, and it is deliberately above the "
            f"median.")


## 7 · Summary

Four things to read.
How many certified, and at what $n_\mathcal{M}$.
The slack
at $n_\mathcal{M}$, since a battery whose median margin is a few bits will not
survive a change of machine.
Which items were certified against the structural
rival alone, which is a real test but a weaker claim than a composer rival at
that prefix.
And $\beta$, the bits the structural rival costs per extra term,
which is the exchange rate between prefix length and unquestionability.


In [ ]:
stage("SUMMARY")
recs = list(DONE.values())
ok  = [r for r in recs if r.get("certified")]
skp = [r for r in recs if r.get("skipped")]
bad = [r for r in recs if not r.get("certified") and not r.get("skipped")]
log(f"{len(recs)} items priced under {TAG}, delta = {DELTA} "
    f"({DELTA_SOURCE}), c(p) unconditional")
_dc = PRICED.get("_delta_calibration")
if _dc:
    log(f"  delta came from the exception clause at {_dc['clause_bits']} bits "
        f"less a margin of {_dc['margin']}. Definition 4.1 requires delta below "
        f"that bound.")
log(f"  {len(ok)} certified, {len(bad)} questionable throughout, {len(skp)} skipped")

if skp:
    for k, v in collections.Counter(r["skipped"] for r in skp).most_common():
        log(f"    {v:4d}  skipped: {k}")

if ok:
    nms = [r["n_M"] for r in ok]
    Ks  = [r["K"] for r in ok]
    sl  = [r["slack"] for r in ok]
    log(f"\n  n_M: median {int(st.median(nms))}, {min(nms)} to {max(nms)}")
    log(f"       {dict(sorted(collections.Counter(nms).items()))}")
    log(f"  K:   median {st.median(Ks):.0f} bits, {min(Ks):.0f} to {max(Ks):.0f}")
    log(f"  slack at n_M: median {st.median(sl):.0f} bits, "
        f"{sum(1 for s in sl if s < 10)} of {len(sl)} within 10 bits of failing")
    so = [r["id"] for r in ok if not r.get("tested_by_composer")]
    if so:
        log(f"\n  {len(so)} of {len(ok)} were certified with only the structural")
        log(f"  rival tested at that prefix. That is a real test, not silence, but")
        log(f"  a composer rival there would be a stronger claim: "
            f"{', '.join(so[:8])}{' ...' if len(so) > 8 else ''}")

if bad:
    log(f"\n  questionable at every tested prefix:")
    for r in bad[:12]:
        est = r.get("L_needed_estimate")
        log(f"    {r['id']}  K={r['K']:.0f}  cheapest structural rival at "
            f"L={r['l_max']} is {r['per_L'][str(r['l_max'])]['struct_cheapest']:.0f}"
            + (f", needs about {est} terms" if est else ""))
    need = [r["L_needed_estimate"] for r in bad if r.get("L_needed_estimate")]
    nofit = [r["id"] for r in bad if not r.get("L_needed_estimate")]
    if need:
        log(f"\n  terms needed, from the fitted line: median "
            f"{int(st.median(need))}, up to {max(need)}")
        near = sum(1 for n in need if n <= (L_MAX_CAP or 60) + 6)
        log(f"  {near} of {len(need)} are within six terms of the current cap of "
            f"{L_MAX_CAP or 'the item length'}")
    if nofit:
        log(f"  {len(nofit)} have no usable estimate, because the cost of the")
        log(f"  structural rival does not grow cleanly with L: "
            f"{', '.join(nofit[:6])}")
    log(f"\n  Two ways to certify an item, since the test is K + delta < cost of")
    log(f"  the structural rival at L:")
    log(f"    more terms   - generation and GPU, and capped by L_MAX_CAP and by")
    log(f"                   how many terms the item has")
    log(f"    a cheaper K  - API only, and free. K is the minimum over the")
    log(f"                   equivalents, so more and shorter equivalents lower")
    log(f"                   it directly. At beta around 7 bits per term, 30")
    log(f"                   bits off K is worth four terms.")

rows = [(r, PR.get(r["id"], {})) for r in recs if not r.get("skipped")]
pred = [(r, p.get("terms_needed_estimate"), p.get("terms_available"))
        for r, p in rows
        if p.get("terms_needed_estimate") and p.get("terms_available")]
if pred:
    tab = collections.Counter()
    for r, need, avail in pred:
        tab[("fits" if need <= avail else "short", bool(r.get("certified")))] += 1
    log(f"\n  the generator's estimate against what happened:")
    log(f"    {'estimate':10s} {'certified':>10s} {'not':>6s}")
    for k in ("fits", "short"):
        log(f"    {k:10s} {tab[(k, True)]:10d} {tab[(k, False)]:6d}")
    _wrong = tab[("short", True)]
    if _wrong:
        log(f"    {_wrong} item(s) certified although the estimate said they "
            f"could not. The proxy is pessimistic and the term ladder that")
        log(f"    rests on it should be revisited.")
    if tab[("fits", False)]:
        log(f"    {tab[('fits', False)]} item(s) failed although the estimate "
            f"said they would fit.")

# ---------------------------------------------------------------------------
# Is the shortest program in CHARACTERS also the cheapest in BITS? `bare` is
# picked by len() at generation time and the generator reasons about rule
# length in characters throughout. K is the minimum in bits under M. Nothing
# guarantees they agree, and everything needed to check was priced anyway.
bb = [r for r in recs if r.get("bare_bits") is not None]
if bb:
    agree = sum(1 for r in bb if r.get("bare_is_argmin"))
    gaps = [r["bare_gap_bits"] for r in bb if not r.get("bare_is_argmin")]
    log(f"\n  bare (shortest in characters) against K (cheapest in bits):")
    log(f"    {agree} of {len(bb)} items have the same winner")
    if gaps:
        log(f"    where they differ, bare costs {st.median(gaps):.0f} bits more "
            f"at the median, up to {max(gaps):.0f}")
        log(f"    so the character proxy the generator steers by does not pick "
            f"the program the measure minimises to.")

_clause = (PRICED.get("_delta_calibration") or {}).get("clause_bits")
if _clause:
    log(f"\n  n_M against the tolerance, from the stored margins. The bound is "
        f"the exception clause at {_clause:.1f} bits;")
    log(f"  the margin below it is ours to choose, and this is what it buys:")
    log(f"    {'margin':>7s} {'delta':>6s} {'certified':>10s}   n_M")
    for _m in (0, 10, 20, 25, 29.6, 35, _clause):
        _d = round(_clause - _m, 1)
        _n = [n_M_at(r, _d) for r in recs if not r.get("skipped")]
        _c = [x for x in _n if x]
        log(f"    {_m:7.1f} {_d:6.1f} {len(_c):5d}/{len(_n):<4d} "
            + (f"median {st.median(_c):.0f}" if _c else "-"))
    log(f"  The run above used delta = {DELTA}. Every row here is exact: the "
        f"walk priced every prefix, so nothing is extrapolated.")

betas = [r["beta"] for r in recs if r.get("beta")]
good = [r["beta"] for r in recs if r.get("beta") and r.get("beta_r2", 0) >= BETA_MIN_R2]
if betas:
    log(f"\n  beta, bits the structural rival costs per extra term:")
    log(f"    all fits    median {st.median(betas):5.1f}   "
        f"{min(betas):.1f} to {max(betas):.1f}   (n = {len(betas)})")
    if good:
        log(f"    R2 >= {BETA_MIN_R2}  median {st.median(good):5.1f}   "
            f"{min(good):.1f} to {max(good):.1f}   (n = {len(good)})")
    log(f"  beta decides nothing: at each L the structural rival is priced and")
    log(f"  compared with the budget directly. It is only the exchange rate used")
    log(f"  to say how many terms an item would need, so a poor fit means no")
    log(f"  estimate rather than a bad one.")

if MARK_QUESTIONABLE:
    now = collections.Counter(b["excluded"] for b in ITEMS if b.get("excluded"))
    log(f"\n  battery marks now: {dict(now)}; "
        f"{sum(1 for b in ITEMS if usable(b))} of {len(ITEMS)} usable")
else:
    would = sum(1 for r in recs if verdict(r)[1])
    log(f"\n  MARK_QUESTIONABLE is off. {would} items would have been marked.")

by_cell = collections.defaultdict(lambda: [0, 0])
for b in ITEMS:
    r = DONE.get(b["id"])
    if not r or r.get("skipped"): continue
    c = by_cell[cell_of(b)]; c[1] += 1
    if r.get("certified"): c[0] += 1
if by_cell:
    log(f"\n  certified by cell:")
    for c in sorted(by_cell):
        good, tot = by_cell[c]
        log(f"    r{c[0]}t{c[1]}  {good}/{tot}")

log(f"\ntotal elapsed: {(time.time()-T0)/60:.1f} min, "
    f"{REF.calls} forward passes")
# GPU seconds: wall clock with the machine loaded. Not utilisation, which is
# the honest thing to record when the work is a long series of short passes.
if "REF" in globals() and hasattr(REF, "_loaded_at"):
    _GPU_SECONDS[0] = time.time() - REF._loaded_at
account(units=len(ok), note=f"{TAG}, {len(ok)} certified of {len(recs)}")
log(f"wrote {PRICING}" + (f" and marks into {BATTERY}" if MARK_QUESTIONABLE else "")
    + (f", copied to {OUTDIR}" if OUTDIR != "." else ""))
state_save(f"{BUILD}: priced {len(ok)} of {len(recs)} under {TAG}")
log("\nNext: the Selector draws a sample from the items that certified, and the")
log("Annotator prices h and the components of their cheapest programs.")
